In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos 
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [2]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
         SELECT DISTINCT * FROM SRUBIO.GRADO_OPORTUNO

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grad_opor = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
df_grad_opor[df_grad_opor['Identificacion'] == '1079912427']

,Identificacion,Periodo,ciclo,Semestre_SINU,pensum,Cod_Unidad,AÑO_GRAD_OPORTUNA,SEMESTRE_MEN_GRAD_OPORTUNA,Programa,Regional,...,NIVELESAPROBADOS,TIPO_INGLES_MALLA,INGLES,NUM_IDENTIFICACION_y,Periodo_conv,Estado,Cursando_niveles,NUM_IDENTIFICACION,PENSAMIENTO CUNISTA,PRACTICAS


In [4]:
# # Paso 1: Ordenar por Periodo y Semestre
# df_grad_opor = df_grad_opor.sort_values(by=['Identificacion',  'Semestre_SINU']).reset_index(drop=True)

# # Paso 2: Reasignar Tipo_grado según orden
# df_grad_opor['Tipo_grado_corregido'] = df_grad_opor.groupby(['Identificacion', 'ciclo']).cumcount().map({
#     0: 'Primer Grado Oportuno',
#     1: 'Segundo Grado Oportuno'
# })
# l = df_grad_opor[df_grad_opor['Identificacion'] == '094135594']
# l[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 
#        'Tipo_grado', 'Tipo_grado_corregido']]


# Filtramos solo los registros que tengan "Grado Oportuno"
mask = df_grad_opor['Tipo_grado'].isin([
    'Primer Grado Oportuno',
    'Segundo Grado Oportuno'
])

# Ordenamos solo los que tienen grado oportuno dentro de cada Identificacion
df_grad_opor.loc[mask, 'Tipo_grado_corregido'] = (
    df_grad_opor[mask]
    .sort_values(by=['Identificacion', 'Semestre_SINU'])
    .groupby('Identificacion')
    .cumcount()
    .map({0: 'Primer Grado Oportuno', 1: 'Segundo Grado Oportuno'})
)

# Para el resto de registros, mantenemos el valor original
df_grad_opor['Tipo_grado_corregido'] = df_grad_opor['Tipo_grado_corregido'].fillna(df_grad_opor['Tipo_grado'])

l = df_grad_opor[df_grad_opor['Identificacion'] == '094135594']
l[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 
       'Tipo_grado', 'Tipo_grado_corregido']]


,Identificacion,Periodo,ciclo,Semestre_SINU,Tipo_grado,Tipo_grado_corregido


In [5]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
         select distinct NUMERO,NIVEL,fecha_grado
         from grados.consolidado_grados

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grados = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [6]:
# Crear tabla pivote con 'fecha_grado' como valor
df_pivot = df_grados.pivot_table(index='NUMERO', columns='NIVEL', values='fecha_grado', aggfunc='first', fill_value='NO GRADUADO').reset_index()

In [7]:
df_pivot.head(3)

NIVEL,NUMERO,ESPECIALIZACION,PROFESIONAL,TECNICO,TECNOLOGO
0,1000000479,NO GRADUADO,NO GRADUADO,2025-07-02,2025-11-10
1,1000000496,NO GRADUADO,2024-04-11 00:00:00,2022-12-13 00:00:00,2023-09-27 00:00:00
2,1000000510,NO GRADUADO,2026-04-13,NO GRADUADO,NO GRADUADO


In [8]:
df_pivot[df_pivot['NUMERO'] == '9975736']

NIVEL,NUMERO,ESPECIALIZACION,PROFESIONAL,TECNICO,TECNOLOGO


In [9]:
df_pivot['NUMERO'] = df_pivot['NUMERO'].astype(str)
df_grad_opor['Identificacion'] = df_grad_opor['Identificacion'].astype(str)

In [10]:
df_graduado_oport = pd.merge(df_grad_opor,df_pivot,left_on=['Identificacion'],right_on=['NUMERO'],how='left')

In [11]:
l = df_grad_opor[df_grad_opor['Identificacion'] == '1079912427']
l[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 
       'Tipo_grado', 'Tipo_grado_corregido']]

,Identificacion,Periodo,ciclo,Semestre_SINU,Tipo_grado,Tipo_grado_corregido


In [12]:
df_grad_opor.columns

Index(['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 'pensum',
       'Cod_Unidad', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Programa', 'Regional', 'ESTADO_ACADEMICO', 'ESTADO_IDIOMAS',
       'Tipo_grado', 'OPCION_GRADO', 'NUM_IDENTIFICACION_x',
       'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA', 'INGLES',
       'NUM_IDENTIFICACION_y', 'Periodo_conv', 'Estado', 'Cursando_niveles',
       'NUM_IDENTIFICACION', 'PENSAMIENTO CUNISTA', 'PRACTICAS',
       'Tipo_grado_corregido'],
      dtype='object')

In [13]:
# Supongamos que tienes un DataFrame llamado df
df_graduado_oport[['ESPECIALIZACIÓN','PROFESIONAL', 'TÉCNICO', 'TECNÓLOGO']] = df_graduado_oport[['ESPECIALIZACION','PROFESIONAL', 'TECNICO', 'TECNOLOGO']].fillna('NO GRADUADO')

### Correr Reconocimientos.ipynb

In [14]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
         SELECT DISTINCT NUM_IDENTIFICACION, COD_PROG_FINAL, PENSUM_L, MARCA,Clasificacion_Final FROM SRUBIO.RECONOCIMIENTOS 

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_tip_insc = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [15]:
df_tip_insc.rename(columns={'NUM_IDENTIFICACION':'Identificacion', 'COD_PROG_FINAL' : 'Cod_Unidad', 'PENSUM_L':'pensum'}, inplace=True)

In [16]:
df_tip_insc['Identificacion'] = df_tip_insc['Identificacion'].astype(str)

In [17]:
df_tip_insc_grad_op = pd.merge(df_graduado_oport,df_tip_insc,left_on=['Identificacion','Cod_Unidad','pensum'],right_on=['Identificacion','Cod_Unidad','pensum'],how='left')

In [18]:
df_tip_insc_grad_op.rename(columns={'MARCA':'TIPO'}, inplace=True)

In [19]:
df_tip_insc_grad_op.loc[(df_tip_insc_grad_op['TIPO'].isna()),'TIPO'] = 'NORMAL'

In [20]:
df_tip_insc_grad_op.loc[
    ((df_tip_insc_grad_op['ciclo'] == 'TÉCNICO') & (df_tip_insc_grad_op['TÉCNICO'] != 'NO GRADUADO')) |
    ((df_tip_insc_grad_op['ciclo'] == 'TECNÓLOGO') & (df_tip_insc_grad_op['TECNÓLOGO'] != 'NO GRADUADO')) |
    ((df_tip_insc_grad_op['ciclo'] == 'PROFESIONAL') & (df_tip_insc_grad_op['PROFESIONAL'] != 'NO GRADUADO')),
    'GRADUADO'
] = 'SI'

In [21]:
# df_tip_insc_grad_op = df_tip_insc_grad_op[df_tip_insc_grad_op['GRADUADO']!='SI']

In [22]:
a=df_tip_insc_grad_op[df_tip_insc_grad_op['Periodo']=='24V06']
a['Identificacion'].nunique()

620

### CORRER TABLERO_INGLES.ipynb

In [23]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
         SELECT DISTINCT Identificacion,Periodo as Ultimo_periodo,ciclo FROM SRUBIO.MARCAS_INGLES

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_marcas = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [24]:
df_marcas[df_marcas['Identificacion'] == '1079912427']

,Identificacion,Ultimo_periodo,ciclo
216096,1079912427,22V06,TECNICO
216097,1079912427,23V06,TECNOLOGO
216098,1079912427,25P03,PROFESIONAL
216099,1079912427,25V01,PROFESIONAL


In [25]:
df_fin = pd.merge(df_tip_insc_grad_op,df_marcas,how='left',on=['Identificacion','ciclo'])

In [26]:
a=df_fin[(df_fin['Ultimo_periodo']=='24V06')&(df_fin['INGLES']=='NO CUMPLE')]
a['Identificacion'].nunique()

634

### Estudiantes que faltan

In [27]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
         SELECT DISTINCT NUM_IDENTIFICACION FROM estudiantes_activos
            WHERE COD_PERIODO NOT LIKE '%E%' AND COD_PERIODO NOT LIKE '%I%'

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_ac = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [28]:
df_fl=df_ac[~df_ac['NUM_IDENTIFICACION'].isin(df_fin['Identificacion'])]

In [29]:
df_fl.rename(columns={'NUM_IDENTIFICACION':'Identificacion'},inplace=True)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_3116\4113589027.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fl.rename(columns={'NUM_IDENTIFICACION':'Identificacion'},inplace=True)


In [30]:
df_fin2 = pd.concat([df_fin,df_fl],ignore_index=True) 

In [31]:
df_fin2.columns

Index(['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 'pensum',
       'Cod_Unidad', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Programa', 'Regional', 'ESTADO_ACADEMICO', 'ESTADO_IDIOMAS',
       'Tipo_grado', 'OPCION_GRADO', 'NUM_IDENTIFICACION_x',
       'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA', 'INGLES',
       'NUM_IDENTIFICACION_y', 'Periodo_conv', 'Estado', 'Cursando_niveles',
       'NUM_IDENTIFICACION', 'PENSAMIENTO CUNISTA', 'PRACTICAS',
       'Tipo_grado_corregido', 'NUMERO', 'ESPECIALIZACION', 'PROFESIONAL',
       'TECNICO', 'TECNOLOGO', 'ESPECIALIZACIÓN', 'TÉCNICO', 'TECNÓLOGO',
       'TIPO', 'Clasificacion_Final', 'GRADUADO', 'Ultimo_periodo'],
      dtype='object')

In [32]:
l = df_fin2[df_fin2['Identificacion'] == '1079912427']
l[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 
       'Tipo_grado']]

,Identificacion,Periodo,ciclo,Semestre_SINU,Tipo_grado


In [33]:
df_fin2 = df_fin2[['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 'pensum',
       'Cod_Unidad', 'AÑO_GRAD_OPORTUNA', 'SEMESTRE_MEN_GRAD_OPORTUNA',
       'Programa', 'Regional', 'ESTADO_ACADEMICO', 'ESTADO_IDIOMAS',
       'Tipo_grado', 'OPCION_GRADO', 'NUM_IDENTIFICACION_x',
       'NIVELESAPROBADOS', 'TIPO_INGLES_MALLA', 'INGLES',
       'NUM_IDENTIFICACION_y', 'Periodo_conv', 'Estado', 'Cursando_niveles',
       'NUM_IDENTIFICACION', 'PENSAMIENTO CUNISTA', 'PRACTICAS', 'NUMERO',
       'ESPECIALIZACION', 'PROFESIONAL', 'TECNICO', 'TECNOLOGO',
       'ESPECIALIZACIÓN', 'TÉCNICO', 'TECNÓLOGO', 'TIPO',
       'Clasificacion_Final', 'GRADUADO', 'Tipo_grado_corregido']]

In [34]:
df_fin2 = df_fin2.drop_duplicates()

In [35]:
df_fin2 = df_fin2[~df_fin2['Periodo'].isna()]
df_fin2.shape

(116354, 37)

In [36]:
df_fin2['Identificacion'].nunique()

74031

In [37]:
##df_fin2.to_excel('Graduacion_oportuna_2024-06-10.xlsx',index=False)

In [38]:
df_fin2[df_fin2['Identificacion'] == '1079912427']

,Identificacion,Periodo,ciclo,Semestre_SINU,pensum,Cod_Unidad,AÑO_GRAD_OPORTUNA,SEMESTRE_MEN_GRAD_OPORTUNA,Programa,Regional,...,PROFESIONAL,TECNICO,TECNOLOGO,ESPECIALIZACIÓN,TÉCNICO,TECNÓLOGO,TIPO,Clasificacion_Final,GRADUADO,Tipo_grado_corregido


In [39]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'GRADO_OPORTUNO2'
schema_name = 'SRUBIO'
# Send the data to sql server
df_fin2.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 1000
)

# End connection
engine.dispose()